In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [15]:
event_log_name = "wide"
log_path = f"./.out/eventlogs/{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [16]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [17]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
# if True:
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File wide-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 1078


In [18]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File wide_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to wide_5000_conformance_results.pkl


In [19]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


/tmp/ipykernel_3978859/1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [20]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print(f"Filtered Metrics DataFrame: {event_log_name}")
display(filtered_metrics_df)

Filtered Metrics DataFrame: wide


,support,confidence
"Responded Existence[Activity Y, Activity B] | |",0.0588,0.989899
"Response[Activity Y, Activity B] | |",0.0588,0.989899
"Responded Existence[Activity Z, Activity B] | |",0.0584,0.986486
"Responded Existence[Activity U, Activity O] | |",0.0500,0.984252
"Responded Existence[Activity U, Activity R] | |",0.0500,0.984252
...,...,...
"Chain Precedence[Activity R, Activity X] | |",0.0634,0.892958
"Chain Precedence[Activity N, Activity P] | |",0.1116,0.892800
"Alternate Response[Activity X, Activity O] | |",0.0630,0.887324
"Alternate Precedence[Activity N, Activity Q] | |",0.1106,0.881978


In [21]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity BJ, Activity B] | |	0.0754	0.9947229551451188

In [22]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    # "Responded Existence[Activity BJ, Activity B] | |", # Gigantic
    "Responded Existence[Activity Q, Activity O] | |", # wide
    ]

In [23]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity Q, Activity O] | |    610
dtype: int64
[21, 38, 39, 60, 64, 95, 108, 117, 119, 134, 141, 142, 143, 149, 155, 156, 174, 185, 215, 237, 244, 255, 257, 262, 267, 268, 277, 295, 299, 307, 323, 324, 336, 359, 361, 364, 370, 377, 380, 389, 390, 393, 404, 417, 421, 425, 434, 445, 451, 460, 469, 475, 477, 480, 488, 492, 498, 499, 502, 510, 526, 545, 546, 549, 573, 575, 584, 587, 595, 608, 612, 616, 628, 630, 632, 640, 643, 652, 653, 656, 672, 673, 682, 694, 705, 712, 715, 733, 742, 744, 764, 766, 768, 772, 773, 776, 777, 784, 800, 807, 808, 820, 826, 829, 830, 851, 865, 866, 887, 891, 899, 900, 905, 906, 928, 931, 953, 962, 965, 967, 970, 979, 980, 990, 993, 1027, 1034, 1044, 1045, 1047, 1055, 1058, 1068, 1072, 1081, 1098, 1121, 1123, 1144, 1151, 1170, 1174, 1181, 1182, 1188, 1209, 1216, 1222, 1226, 1227, 1238, 1244, 1249, 1250, 1253, 1254, 1265, 1270, 1278, 1282, 1326, 1328, 1335, 1340, 1347, 1355, 1381, 1395, 1416, 1418, 1437, 1438, 1440, 1446, 1448, 1458, 1463, 

In [24]:
print("END")

END


In [25]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)